# Exploratory Data Analysis — Data Quality Analysis

Member 3 - Mathuran: Data Quality section of `02_eda.ipynb`.

Dataset: Kaggle Playground Series S4E2 — Obesity Risk Prediction (https://www.kaggle.com/competitions/playground-series-s4e2/data). Raw file: `data/raw/obesity.csv`. The raw dataset is inspected without modifying it.

## Objectives (Member 3 scope only)

- Analyse missing values
- Check duplicate records
- Verify data types
- Analyse statistical properties
- Identify dataset issues requiring preprocessing and state how each should be handled

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "obesity.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

In [ ]:
identifier_column = "id"
target_column = "NObeyesdad"

numerical_features = [
    "Age",
    "Height",
    "Weight",
    "FCVC",
    "NCP",
    "CH2O",
    "FAF",
    "TUE",
]

categorical_features = [
    "Gender",
    "family_history_with_overweight",
    "FAVC",
    "CAEC",
    "SMOKE",
    "SCC",
    "CALC",
    "MTRANS",
]

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

## 1. Missing values

Every cell is checked for `NaN`. The count and percentage per column show whether any imputation is required before preprocessing.

In [ ]:
missing_counts = df.isnull().sum()
missing_percentages = (df.isnull().mean().mul(100).round(2))

missing_summary = pd.DataFrame({
    "Missing count": missing_counts,
    "Missing percentage": missing_percentages,
})

print("Total missing cells:", int(missing_counts.sum()))
display(missing_summary)

In [ ]:
plt.figure(figsize=(10, 4))

sns.barplot(
    x=missing_summary.index,
    y="Missing percentage",
    data=missing_summary.reset_index().rename(columns={"index": "column"}),
)

plt.title("Missing Values by Column (%)")
plt.xlabel("Column")
plt.ylabel("Missing (%)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 2. Duplicate records

Exact duplicate rows would inflate the effective sample size, and a duplicated `id` would break the assumption that the identifier uniquely identifies one record. Both are checked.

In [ ]:
exact_duplicates = int(df.duplicated().sum())
feature_duplicates = int(
    df.duplicated(subset=[column for column in df.columns if column != identifier_column]).sum()
)

print("Exact duplicate rows:", exact_duplicates)
print("Duplicate rows excluding id:", feature_duplicates)
print("id is unique:", bool(df[identifier_column].is_unique))
print("id range:", int(df[identifier_column].min()), "to", int(df[identifier_column].max()))
print("id unique count:", int(df[identifier_column].nunique()))

## 3. Data types

Pandas dtypes are compared against the expected Kaggle schema: `id` is an integer identifier, the eight lifestyle/physical measures are floating point numbers, and the remaining columns (including the target) are strings. A mismatch would signal a parsing problem that must be fixed before encoding or scaling.

In [ ]:
print(df.dtypes)
print()

for feature in numerical_features:
    print(feature, "->", df[feature].dtype)

print()

for feature in categorical_features + [target_column]:
    print(feature, "->", df[feature].dtype)

In [ ]:
for column in categorical_features + [target_column]:
    category_counts = df[column].value_counts(dropna=False)
    category_percentages = (
        df[column]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .round(2)
    )

    category_summary = pd.DataFrame({
        "Count": category_counts,
        "Percentage": category_percentages,
    })

    print(f"\n{column}")
    print("-" * len(column))
    display(category_summary)

## 4. Statistical properties


In [ ]:
display(df[numerical_features].describe().T.round(2))

In [ ]:
shape_summary = pd.DataFrame({
    "Skewness": df[numerical_features].skew().round(3),
    "Kurtosis": df[numerical_features].kurt().round(3),
})

display(shape_summary)

In [ ]:
first_quartile = df[numerical_features].quantile(0.25)
third_quartile = df[numerical_features].quantile(0.75)
interquartile_range = third_quartile - first_quartile
lower_bound = first_quartile - 1.5 * interquartile_range
upper_bound = third_quartile + 1.5 * interquartile_range
outlier_counts = ((df[numerical_features] < lower_bound) | (df[numerical_features] > upper_bound)).sum()
outlier_percentages = (outlier_counts / len(df) * 100).round(2)
outlier_summary = pd.DataFrame({"IQR outlier count": outlier_counts, "IQR outlier percentage": outlier_percentages})
display(outlier_summary)

In [ ]:
figure, axes = plt.subplots(nrows=2, ncols=4, figsize=(14, 7))
for axis, column in zip(axes.flat, numerical_features):
    sns.boxplot(data=df, x=column, ax=axis)
    axis.set_title(f"Boxplot of {column}")
    axis.set_xlabel(column)
plt.tight_layout()
plt.show()

## 5. Validity, sparsity and target balance


In [ ]:
print("Numerical minima:")
print(df[numerical_features].min().round(2).to_string())
print("Empty-string cells:", int((df.astype(str) == "").sum().sum()))
print("Target missing values:", int(df[target_column].isnull().sum()))
print("Target unique classes:", int(df[target_column].nunique()))

In [ ]:
target_counts = df[target_column].value_counts()
target_percentages = (df[target_column].value_counts(normalize=True).mul(100).round(2))
target_summary = pd.DataFrame({"Count": target_counts, "Percentage": target_percentages})
display(target_summary)

## 6. Data-quality issues and preprocessing handoff (Member 3 conclusion)

- Missing: 0 NaN everywhere, no imputation needed.
- Duplicates: 0 rows; id unique 0-20757; drop id before training.
- Types: int64 / float64 / strings as expected.
- Shape: Age skew 1.59 (1074 IQR flags); NCP skew -1.56 (6052 flags); do not delete, scale instead.
- Rare levels: SMOKE=yes 245, SCC=yes 687, Bike 32, Motorbike 38, CAEC=no 279; one-hot with unknown handling.
- Target: 7 classes 2427 to 4046; use stratified split.
- Leakage: fit preprocessors on train only; exclude id and target.
